# ExternalPatch demo — Filament formation (P–D–P rods)

## Overview

**ExternalPatch** is a 4-body "external patch" force for HOOMD-blue.
Each patched particle *i* has a partner *j* that defines the patch direction
$\hat p = (\mathbf{r}_i - \mathbf{r}_j)/|\cdots|$ (pointing *outward*).
The interaction between two patched particles *i* and *k* is:

$$
U_{ik} = f_i\,f_k\;\varepsilon\!\left(1 - r^2/r_c^2\right)^2
$$

with cubic Hermite (smoothstep) angular envelopes
$f = S\!\left(\frac{\hat p \cdot \hat r - (1 - w)}{w}\right)$, where
$S(t) = 3t^2 - 2t^3$ for $0 \le t \le 1$.

## Filament geometry

Each rod is a **P–D–P** tripole.  The central D particle is bonded to
both end P particles.  A harmonic angle keeps the rod straight.

Both P particles carry patches pointing outward (away from D).
When two P-ends from different rods approach each other,
their outward-pointing patches face *toward* each other → attraction
→ the rods link end-to-end into **filaments**.

> **Patch direction convention:**
> `ExternalPatch` computes the patch direction as
> $\hat p = (\text{particle\_pos} - \text{partner\_pos})/|\cdots|$.
> With D at the centre, both P patches point *outward* (away from D).
> When two P-ends from different rods approach each other, their
> outward-pointing patches face *toward* each other → attraction.

| Parameter | Value | Notes |
|-----------|-------|-------|
| N_tripoles | 500 | 1 500 particles total |
| Box | L × L × 5 | Flat 3D slab |
| ε | 30 | ~30 kT → stable contacts |
| width | 0.5 | Hermite transition width |
| r\_cut (patch) | 1.5 | Patch attraction range |
| A (DPD, P–P) | 40 | Soft repulsion |
| γ (DPD) | 1.0 | DPD friction |
| r\_cut (DPD) | 1.0 | DPD cutoff |
| Bond k (P–D) | 200 | Director tether |
| Bond r₀ | 0.5 | P–D bond length |
| Angle k | 100 | Straightness stiffness |
| Angle t₀ | π | Equilibrium = straight |
| kT | 1.0 | Thermal energy |

In [ ]:
import sys
sys.path.insert(0, "/groups/goloborodko/user/anton.goloborodko/src/hoomd-blue/build/install_mixed/lib/python3.12/site-packages")

import numpy as np
import matplotlib.pyplot as plt
import hoomd
from hoomd import align_angle

print("HOOMD version:", hoomd.version.version)



## 1. Build the initial configuration

Place 500 tripole rods on a jittered grid in a flat 3D box (Lz = 5).
Each rod = 3 particles: P\_a, D, P\_b with random 3D orientation.

In [ ]:

N_tripoles = 500
N = 3 * N_tripoles          # 1500 particles
bond_r0 = 0.5               # P-D bond length
Lz = 5.0                    # thin slab
grid_side = int(np.ceil(np.sqrt(N_tripoles)))  # 23
spacing = 1.3
L = grid_side * spacing + 4.0

device = hoomd.device.auto_select()
snap = hoomd.Snapshot(device.communicator)

if snap.communicator.rank == 0:
    snap.configuration.box = [L, L, Lz, 0, 0, 0]
    snap.particles.N = N
    snap.particles.types = ["P", "D"]
    snap.particles.mass[:] = 1.0

    rng = np.random.default_rng(42)
    positions = np.zeros((N, 3))
    typeids = np.zeros(N, dtype=int)

    for idx in range(N_tripoles):
        row = idx // grid_side
        col = idx % grid_side
        cx = (col - grid_side / 2) * spacing + rng.uniform(-0.3, 0.3)
        cy = (row - grid_side / 2) * spacing + rng.uniform(-0.3, 0.3)
        cz = rng.uniform(-Lz / 2 + 0.5, Lz / 2 - 0.5)

        # Random 3D rod direction (uniform on sphere)
        cos_th = rng.uniform(-1, 1)
        sin_th = np.sqrt(1 - cos_th**2)
        phi = rng.uniform(0, 2 * np.pi)
        dx = sin_th * np.cos(phi)
        dy = sin_th * np.sin(phi)
        dz = cos_th

        i_Pa = 3 * idx       # P_a
        i_D  = 3 * idx + 1   # D (centre)
        i_Pb = 3 * idx + 2   # P_b

        # Layout: P_a -- D (centre) -- P_b
        positions[i_D]  = [cx, cy, cz]
        positions[i_Pa] = [cx - bond_r0 * dx, cy - bond_r0 * dy, cz - bond_r0 * dz]
        positions[i_Pb] = [cx + bond_r0 * dx, cy + bond_r0 * dy, cz + bond_r0 * dz]

        typeids[i_Pa] = 0  # P
        typeids[i_D]  = 1  # D
        typeids[i_Pb] = 0  # P

    # Wrap positions into the periodic box
    positions[:, 0] -= L * np.round(positions[:, 0] / L)
    positions[:, 1] -= L * np.round(positions[:, 1] / L)
    positions[:, 2] -= Lz * np.round(positions[:, 2] / Lz)

    snap.particles.position[:] = positions
    snap.particles.typeid[:] = typeids
    snap.particles.velocity[:] = rng.normal(0, 0.5, (N, 3))

    # Bonds: 2 per tripole  (P_a-D and D-P_b)
    snap.bonds.N = 2 * N_tripoles
    snap.bonds.types = ["PD"]
    snap.bonds.typeid[:] = 0
    for idx in range(N_tripoles):
        i_Pa = 3 * idx
        i_D  = 3 * idx + 1
        i_Pb = 3 * idx + 2
        b = 2 * idx
        snap.bonds.group[b]     = [i_Pa, i_D]    # P_a - D
        snap.bonds.typeid[b]    = 0
        snap.bonds.group[b + 1] = [i_D, i_Pb]    # D - P_b
        snap.bonds.typeid[b + 1] = 0

    # Angles: 1 per tripole -- keep rod straight (P_a - D - P_b, t0 = pi)
    snap.angles.N = N_tripoles
    snap.angles.types = ["PDP"]
    snap.angles.typeid[:] = 0
    for idx in range(N_tripoles):
        i_Pa = 3 * idx
        i_D  = 3 * idx + 1
        i_Pb = 3 * idx + 2
        snap.angles.group[idx] = [i_Pa, i_D, i_Pb]

print(f"N_tripoles = {N_tripoles}, N_particles = {N}, L = {L:.1f}, Lz = {Lz}")


## 2. Set up forces and integrator


In [ ]:
sim = hoomd.Simulation(device=device, seed=42)
sim.create_state_from_snapshot(snap)

# ── DPD (conservative + dissipative + random, thermostat included) ────────
nlist_dpd = hoomd.md.nlist.Cell(buffer=0.4)
dpd = hoomd.md.pair.DPD(nlist=nlist_dpd, default_r_cut=1.0, kT=1.0)
dpd.params[("P", "P")] = dict(A=40.0, gamma=1.0)
dpd.params[("P", "D")] = dict(A=0.0, gamma=1.0)
dpd.params[("D", "D")] = dict(A=0.0, gamma=1.0)

bond_force = hoomd.md.bond.Harmonic()
bond_force.params["PD"] = dict(k=200.0, r0=bond_r0)

angle_force = hoomd.md.angle.Harmonic()
angle_force.params["PDP"] = dict(k=100.0, t0=np.pi)  # keep rods straight

nlist_patch = hoomd.md.nlist.Cell(buffer=0.4)
patch = align_angle.ExternalPatch(nlist=nlist_patch, r_cut=1.5)
patch.epsilon = 30.0
patch.width   = 0.5
partners = []
for i in range(N_tripoles):
    partners.append((3 * i,     3 * i + 1))   # P_a -> D
    partners.append((3 * i + 2, 3 * i + 1))   # P_b -> D  (same D!)
patch.partners = partners

# ── NVE integrator (DPD provides the thermostat) ─────────────────────────
method = hoomd.md.methods.ConstantVolume(filter=hoomd.filter.All())

integrator = hoomd.md.Integrator(dt=0.005,
                                  forces=[dpd, bond_force, angle_force, patch],
                                  methods=[method])
sim.operations.integrator = integrator

thermo = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
sim.operations.computes.append(thermo)

print("Forces: DPD (PP A=40), Harmonic bonds (PD k=200),")
print("        Harmonic angle (PDP k=100, t0=pi),")
print("        ExternalPatch (eps=30, width=0.5, r_cut=1.5)")


## 3. Equilibrate and save snapshots

Run 500 000 steps (2 500 τ) total.  Save snapshots at the start,
midpoint, and end.


In [ ]:

snapshots = {}

sim.run(0)
snapshots["t = 0"] = sim.state.get_snapshot()
ke = thermo.kinetic_energy / N
print(f"t = 0:     PE = {thermo.potential_energy:.1f},  KE/particle = {ke:.3f}")

sim.run(250_000)
snapshots["t = 1250 \u03c4"] = sim.state.get_snapshot()
ke = thermo.kinetic_energy / N
print(f"t = 1250\u03c4: PE = {thermo.potential_energy:.1f},  KE/particle = {ke:.3f}")

sim.run(250_000)
snapshots["t = 2500 \u03c4"] = sim.state.get_snapshot()
ke = thermo.kinetic_energy / N
print(f"t = 2500\u03c4: PE = {thermo.potential_energy:.1f},  KE/particle = {ke:.3f}")



## 4. Visualize filament formation

x–y projection of the flat box.  Each panel shows:
- **Green** circles: P particles with an inter-rod contact
- **Red** circles: P particles without inter-rod contacts
- **Orange** dots + tethers: D (director) particles
- **Grey** lines: intra-rod P–P bonds (rod axis)
- **Dark green** lines: inter-rod P–P contacts (filament bonds)

A **zoomed-in** panel shows a random 10 × 10 region.


In [ ]:

def find_filaments(pos_P, box, N_tripoles, cutoff=1.3):
    """Find inter-tripole P-P contacts and connected-component filaments."""
    n = len(pos_P)
    Lx, Ly, Lz = box
    cutsq = cutoff * cutoff
    contacts = []
    tripole_adj = {}

    for i in range(n):
        ti = i // 2
        for j in range(i + 1, n):
            tj = j // 2
            if ti == tj:
                continue
            dx = pos_P[j, 0] - pos_P[i, 0]
            dy = pos_P[j, 1] - pos_P[i, 1]
            dz = pos_P[j, 2] - pos_P[i, 2]
            dx -= Lx * np.round(dx / Lx)
            dy -= Ly * np.round(dy / Ly)
            dz -= Lz * np.round(dz / Lz)
            dsq = dx * dx + dy * dy + dz * dz
            if dsq < cutsq:
                contacts.append((i, j))
                tripole_adj.setdefault(ti, set()).add(tj)
                tripole_adj.setdefault(tj, set()).add(ti)

    visited = set()
    filaments = []
    for t in range(N_tripoles):
        if t in visited:
            continue
        component = []
        queue = [t]
        while queue:
            node = queue.pop(0)
            if node in visited:
                continue
            visited.add(node)
            component.append(node)
            for nb in tripole_adj.get(node, set()):
                if nb not in visited:
                    queue.append(nb)
        filaments.append(component)
    return contacts, filaments


def plot_state(ax, snapshot, title, box, N_tripoles, cutoff=1.3, zoom=None):
    """Plot tripole rods with filament bonds (x-y projection)."""
    pos = np.array(snapshot.particles.position)
    tid = np.array(snapshot.particles.typeid)
    Lx, Ly, Lz = box

    pos_P = pos[tid == 0]
    pos_D = pos[tid == 1]

    contacts, filaments = find_filaments(pos_P, box, N_tripoles, cutoff)

    bonded_P = set()
    for i, j in contacts:
        bonded_P.add(i)
        bonded_P.add(j)

    if zoom is not None:
        s_P, s_D = 40, 10
        lw_inter, lw_intra, lw_tether = 3.5, 1.5, 1.0
    else:
        s_P, s_D = 6, 2
        lw_inter, lw_intra, lw_tether = 1.2, 0.4, 0.25

    # Intra-rod P_a -- D -- P_b (grey rod axis)
    for k in range(N_tripoles):
        ia, ib = 2 * k, 2 * k + 1  # P_a and P_b indices in pos_P
        x_Pa, y_Pa = pos_P[ia, 0], pos_P[ia, 1]
        x_Pb, y_Pb = pos_P[ib, 0], pos_P[ib, 1]
        dx = x_Pb - x_Pa
        dy = y_Pb - y_Pa
        dx -= Lx * np.round(dx / Lx)
        dy -= Ly * np.round(dy / Ly)
        ax.plot([x_Pa, x_Pa + dx],
                [y_Pa, y_Pa + dy],
                color="silver", lw=lw_intra, alpha=0.7, zorder=1)

    # D particles (orange dots at rod centres)
    ax.scatter(pos_D[:, 0], pos_D[:, 1], c="orange", s=s_D, zorder=2,
               edgecolors="none", alpha=0.5)

    colors = ["green" if i in bonded_P else "red" for i in range(len(pos_P))]
    ax.scatter(pos_P[:, 0], pos_P[:, 1], c=colors, s=s_P, zorder=3,
               edgecolors="none")

    for i, j in contacts:
        dx = pos_P[j, 0] - pos_P[i, 0]
        dy = pos_P[j, 1] - pos_P[i, 1]
        dx -= Lx * np.round(dx / Lx)
        dy -= Ly * np.round(dy / Ly)
        ax.plot([pos_P[i, 0], pos_P[i, 0] + dx],
                [pos_P[i, 1], pos_P[i, 1] + dy],
                color="darkgreen", lw=lw_inter, alpha=0.85, zorder=4,
                solid_capstyle="round")

    in_fil = sum(1 for f in filaments if len(f) >= 2 for _ in f)
    frac = in_fil / N_tripoles
    ax.set_title(f"{title}\n{len(contacts)} bonds, {in_fil}/{N_tripoles} rods linked ({frac:.0%})",
                 fontsize=10)

    if zoom is not None:
        cx, cy, hw = zoom
        ax.set_xlim(cx - hw, cx + hw)
        ax.set_ylim(cy - hw, cy + hw)
    else:
        ax.set_xlim(-Lx / 2, Lx / 2)
        ax.set_ylim(-Ly / 2, Ly / 2)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=8)


box_dims = (L, L, Lz)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (label, snap_i) in zip(axes, snapshots.items()):
    plot_state(ax, snap_i, label, box_dims, N_tripoles)
fig.suptitle("Filament formation in a P-D-P tripole gas (x-y projection)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

rng_zoom = np.random.default_rng(123)
half_box = L / 2
zoom_hw = 5.0
zx = rng_zoom.uniform(-half_box + zoom_hw, half_box - zoom_hw)
zy = rng_zoom.uniform(-half_box + zoom_hw, half_box - zoom_hw)

fig_z, ax_z = plt.subplots(figsize=(8, 8))
plot_state(ax_z, snapshots["t = 2500 \u03c4"], "Zoomed in (final state)",
           box_dims, N_tripoles, zoom=(zx, zy, zoom_hw))
plt.tight_layout()
plt.show()


## 5. Filament length distribution & NN distances

- **Filament sizes** — connected components of tripoles linked by
  inter-rod P–P contacts (cutoff 1.3).
- **NN distance** — nearest P–P distance across different tripoles
  for each P particle.


In [ ]:

final_snap = snapshots["t = 2500 \u03c4"]
pos = np.array(final_snap.particles.position)
tid = np.array(final_snap.particles.typeid)
pos_P = pos[tid == 0]

contacts, filaments = find_filaments(pos_P, (L, L, Lz), N_tripoles, cutoff=1.3)

fil_sizes = sorted([len(f) for f in filaments], reverse=True)
n_in_fil = sum(s for s in fil_sizes if s >= 2)

print(f"Total tripoles: {N_tripoles}")
print(f"Tripoles in filaments (size >= 2): {n_in_fil} ({n_in_fil / N_tripoles:.0%})")
print(f"Number of filaments (size >= 2): {sum(1 for s in fil_sizes if s >= 2)}")
print(f"Longest filament: {fil_sizes[0]} tripoles")
print(f"Mean filament size (>=2): {np.mean([s for s in fil_sizes if s >= 2]):.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

sizes_ge2 = [s for s in fil_sizes if s >= 2]
if sizes_ge2:
    ax1.hist(sizes_ge2, bins=range(2, max(sizes_ge2) + 2), color="seagreen",
             edgecolor="white", linewidth=0.5, align="left")
ax1.set_xlabel("Filament size (tripoles)", fontsize=12)
ax1.set_ylabel("Count", fontsize=12)
ax1.set_title("Filament size distribution", fontsize=13)

n_P = len(pos_P)
nn_dists = []
for i in range(n_P):
    ti = i // 2
    best_dsq = np.inf
    for j in range(n_P):
        if j == i or j // 2 == ti:
            continue
        dx = pos_P[j, 0] - pos_P[i, 0]
        dy = pos_P[j, 1] - pos_P[i, 1]
        dz = pos_P[j, 2] - pos_P[i, 2]
        dx -= L * np.round(dx / L)
        dy -= L * np.round(dy / L)
        dz -= Lz * np.round(dz / Lz)
        dsq = dx * dx + dy * dy + dz * dz
        if dsq < best_dsq:
            best_dsq = dsq
    nn_dists.append(np.sqrt(best_dsq))
nn_dists = np.array(nn_dists)

ax2.hist(nn_dists, bins=60, range=(0, 4), color="steelblue",
         edgecolor="white", linewidth=0.3)
ax2.axvline(1.3, color="red", ls="--", lw=1, label="contact cutoff (1.3)")
ax2.set_xlabel("Nearest inter-rod P-P distance", fontsize=12)
ax2.set_ylabel("Count", fontsize=12)
ax2.set_title("NN distance distribution (final state)", fontsize=13)
ax2.legend()

plt.tight_layout()
plt.show()

bonded_frac = np.mean(nn_dists < 1.3)
print(f"\nFraction of P particles with inter-rod NN < 1.3: {bonded_frac:.1%}")



## Summary

With the P–D–P architecture (shared D directing both P ends), each
rod has two **inward-pointing** patches.  When P ends from different
rods approach face-to-face, their inward patches align toward each
other, enabling end-to-end binding and filament formation.

**To experiment:**
- Widen α → 1.0  to allow branching / sheet formation
- Lower ε → 5    to see transient filaments that break and reform
- Increase N      for longer filaments at higher density